# 16S Analyses of the Longitudinal Acne Study
## V1-V3 and V4 primer sets comparison

Date created: 12/28/2024

Notebook authors: Yang Chen

Data analysis by: Yang Chen, Tyler Myers, Britta De Pessemier

This notebook plots the following:

- Plot showing abundance of Cutibacterium in each sample with each primer pair (i.e. the axes are V13 vs V4, each point is the amount of Cutibacterium in one sample by each of the primer pairs)
- Venn diagram illustrating the overlap of genera-level taxa detected by both primer sets, alongside those unique to V1-V3 or V4

In [171]:
# Import Python packages
import pandas as pd
import numpy as np
import biom
import matplotlib.pyplot as plt
import seaborn as sns
import gemelli
from gemelli.preprocessing import rclr_transformation
from matplotlib_venn import venn2

In [172]:
# Function to load BIOM table, collapse by taxa, sort rows by row sum, remove specified samples, and convert to relative abundance
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    return df


In [173]:
# Read and transform V1-V3 and V4 Genus level absolute abundance table
V1V3_biom = biom_to_df('../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom')
V4_biom = biom_to_df('../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom')

In [174]:
# Extract the row corresponding to 'g__Cutibacterium' for V1-V3
V1_V3_cutibacterium_df = V1V3_biom.loc[[' g__Cutibacterium']]

# Rename the row index
V1_V3_cutibacterium_df.index = [' g__Cutibacterium V1-V3']

# Transpose the DataFrame
V1_V3_cutibacterium_df = V1_V3_cutibacterium_df.T

# Display the transformed DataFrame
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3
LAMI.RD001.D0.C1,2362.0
LAMI.RD001.D0.C2,1808.0
LAMI.RD001.D14.C1,2234.0
LAMI.RD001.D14.C2,1761.0
LAMI.RD001.D28.C1,2367.0
...,...
LAMI.RD319.D14.C1,1900.0
LAMI.RD319.D21.C3,1003.0
LAMI.RD319.D14.C2,2315.0
LAMI.RD319.D9.C1,1230.0


In [175]:
# Extract the row corresponding to 'g__Cutibacterium' for V4
V4_cutibacterium_df = V4_biom.loc[[' g__Cutibacterium']]

# Rename the row index
V4_cutibacterium_df.index = [' g__Cutibacterium V4']

# Transpose the DataFrame
V4_cutibacterium_df = V4_cutibacterium_df.T

# Display the transformed DataFrame
V4_cutibacterium_df

,g__Cutibacterium V4
LAMI.RD001.D0.C1,1.0
LAMI.RD001.D14.C1,0.0
LAMI.RD004.D0.C2,1.0
LAMI.RD001.D0.C2,3.0
LAMI.RD004.D28.C1,0.0
...,...
LAMI.RD319.D16.C2,3.0
LAMI.RD319.D28.C1,0.0
LAMI.RD318.D9.C2,10.0
LAMI.RD319.D28.C2,0.0


In [176]:
# Get ranks for V1-V3
V1_V3_cutibacterium_df['V1-V3'] = V1_V3_cutibacterium_df[' g__Cutibacterium V1-V3'].rank(method='min').astype(int)
V1_V3_cutibacterium_df

,g__Cutibacterium V1-V3,V1-V3
LAMI.RD001.D0.C1,2362.0,31
LAMI.RD001.D0.C2,1808.0,12
LAMI.RD001.D14.C1,2234.0,25
LAMI.RD001.D14.C2,1761.0,11
LAMI.RD001.D28.C1,2367.0,32
...,...,...
LAMI.RD319.D14.C1,1900.0,15
LAMI.RD319.D21.C3,1003.0,4
LAMI.RD319.D14.C2,2315.0,29
LAMI.RD319.D9.C1,1230.0,5


In [177]:
# Get ranks for V4
V4_cutibacterium_df['V4'] = V4_cutibacterium_df[' g__Cutibacterium V4'].rank(method='min').astype(int)
V4_cutibacterium_df

,g__Cutibacterium V4,V4
LAMI.RD001.D0.C1,1.0,12
LAMI.RD001.D14.C1,0.0,1
LAMI.RD004.D0.C2,1.0,12
LAMI.RD001.D0.C2,3.0,42
LAMI.RD004.D28.C1,0.0,1
...,...,...
LAMI.RD319.D16.C2,3.0,42
LAMI.RD319.D28.C1,0.0,1
LAMI.RD318.D9.C2,10.0,79
LAMI.RD319.D28.C2,0.0,1


In [178]:
# Concatenate the two DataFrames along columns, matching on indexes
# Select specific columns
v1_v3_column = V1_V3_cutibacterium_df['V1-V3']  # Adjust column name if needed
v4_column = V4_cutibacterium_df['V4']  # Adjust column name if needed

# Concatenate the selected columns
combined_cutibacterium_df = pd.concat([v1_v3_column, v4_column], axis=1)

# Rename columns for clarity (optional)
combined_cutibacterium_df.columns = ['V1-V3', 'V4']

# Drop rows with NaN values
combined_cutibacterium_df = combined_cutibacterium_df.dropna()

combined_cutibacterium_df

,V1-V3,V4
LAMI.RD001.D0.C1,31.0,12.0
LAMI.RD001.D0.C2,12.0,42.0
LAMI.RD001.D14.C1,25.0,1.0
LAMI.RD001.D14.C2,11.0,1.0
LAMI.RD001.D28.C1,32.0,12.0
...,...,...
LAMI.RD318.D28.C3,77.0,51.0
LAMI.RD319.D14.C1,15.0,12.0
LAMI.RD319.D14.C2,29.0,1.0
LAMI.RD319.D9.C1,5.0,1.0


In [179]:
# Scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(
    combined_cutibacterium_df.iloc[:, 0], 
    combined_cutibacterium_df.iloc[:, 1], 
    color='#ffa505', 
    edgecolor='black',
    linewidth=0.5,
    alpha=0.7, 
    label='Samples'
)

# Best-fit line
x = combined_cutibacterium_df.iloc[:, 0]
y = combined_cutibacterium_df.iloc[:, 1]
m, b = np.polyfit(x, y, 1)  # Linear regression
correlation = combined_cutibacterium_df.corr().iloc[0, 1]
plt.plot(x, m * x + b, color='darkorange', label=f'Correlation: {correlation:.2f}')

# Title and labels
plt.title('Per-sample correlation between V1-V3 and V4 by Cutibacterium', fontsize=14)
plt.xlabel('Ranked Abundance of Cutibacterium (V1-V3)', fontsize=10)
plt.ylabel('Ranked Abundance of Cutibacterium (V4)', fontsize=10)

# Add correlation coefficient
correlation = combined_cutibacterium_df.corr().iloc[0, 1]

# Grid, legend, and save
plt.grid(True)
plt.legend()
plt.savefig('../Figures/16S_Figures/primer_comparison/Cutbacterium_V1V3_vs_V4_correlation_ranks.png', dpi=600)
plt.show()


/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_48337/1018578260.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Venn diagram comparing genera-level taxa between V1-V3 and V4

In [198]:
# Convert indices to sets for set operations
indices_V1V3 = set(V1V3_biom.index)
indices_V4 = set(V4_biom.index)

# Taxa unique to V1V3
unique_to_V1V3 = indices_V1V3 - indices_V4

# Taxa unique to V4
unique_to_V4 = indices_V4 - indices_V1V3

# Taxa shared by both
shared_taxa = indices_V1V3 & indices_V4

# Step 2: Filter shared and unique taxa based on 10% sample prevalence
def filter_by_prevalence(df, taxa, prevalence_threshold=0.1):
    # Subset the DataFrame to include only the specified taxa
    subset_df = df.loc[list(taxa)]  # Convert set to list for indexing
    # Calculate prevalence: proportion of samples where the taxon is present
    prevalence = (subset_df > 0).sum(axis=1) / subset_df.shape[1]
    # Filter taxa based on the prevalence threshold
    return set(prevalence[prevalence >= prevalence_threshold].index)

# Apply prevalence filtering
filtered_shared = filter_by_prevalence(V1V3_biom, shared_taxa, prevalence_threshold=0.1)
filtered_unique_to_V1V3 = filter_by_prevalence(V1V3_biom, unique_to_V1V3, prevalence_threshold=0.1)
filtered_unique_to_V4 = filter_by_prevalence(V4_biom, unique_to_V4, prevalence_threshold=0.1)

# Step 3: Create the Venn diagram with enhancements
plt.clf()
plt.figure(figsize=(8, 8))  # Increased figure size for better clarity

# Create the Venn diagram with customized circle outlines
venn = venn2(
    [filtered_unique_to_V1V3 | filtered_shared, filtered_unique_to_V4 | filtered_shared],
    ('', ''),
    set_colors=('lightblue', 'lightgreen'),  # Fill colors for the circles
    alpha=0.4  # Transparency for fill colors
)

# Customize the circle outlines with darker colors
for circle, color in zip(['10', '01'], ['blue', 'green']):
    venn.get_patch_by_id(circle).set_edgecolor(color)  # Darker outline color
    venn.get_patch_by_id(circle).set_linewidth(2)  # Thickness of the outline

# Customizing colors for the Venn diagram (switching green and purple)
venn.get_patch_by_id('10').set_color('#87CEEB')  # Light blue for V1-V3
venn.get_patch_by_id('01').set_color('#DDA0DD')  # Light purple for V4
venn.get_patch_by_id('11').set_color('#98FB98')  # Light green for shared

# Customizing text labels
for id in ['10', '01', '11']:
    if venn.get_label_by_id(id):
        venn.get_label_by_id(id).set_fontsize(12)  # Larger font size
        venn.get_label_by_id(id).set_color('black')  # Black text for better readability

# Add a title with larger font size and weight
plt.title('Bacterial Genera Resolved by 16S V1-V3 vs V4',
          fontsize=18)

# Add a legend for the groups
plt.legend(
    handles=[
        plt.Line2D([0], [0], marker='o', color='#87CEEB', lw=0, label='V1-V3 Unique'),
        plt.Line2D([0], [0], marker='o', color='#98FB98', lw=0, label='Shared'),
        plt.Line2D([0], [0], marker='o', color='#DDA0DD', lw=0, label='V4 Unique'),
    ],
    loc='lower center', bbox_to_anchor=(0.5, -0.075), ncol=3, fontsize=12
)

# Save the figure with a higher resolution
plt.savefig('../Figures/16S_Figures/primer_comparison/primer_venn_diagram.png', dpi=600, bbox_inches='tight')

# Show the figure (optional)
plt.show()

# Print the results
print("Filtered Unique to V1V3 (>=10% prevalence):")
print(filtered_unique_to_V1V3)

print("\nFiltered Unique to V4 (>=10% prevalence):")
print(filtered_unique_to_V4)

print("\nFiltered Shared Taxa (>=10% prevalence):")
print(filtered_shared)


Filtered Unique to V1V3 (>=10% prevalence):
{' g__Janibacter', ' g__Reyranella', ' g__Mycobacterium'}

Filtered Unique to V4 (>=10% prevalence):
{' g__Empedobacter', ' g__Stenotrophomonas', ' g__Capnocytophaga', ' g__Aerococcus', ' g__Luteimonas', ' g__Aggregatibacter', ' g__Peptostreptococcus', ' g__Enhydrobacter', ' g__Abiotrophia', ' g__Bifidobacterium', ' g__Geobacillus', ' g__Psychrobacter', ' g__Massilia', ' g__Bradyrhizobium', ' g__Peptoniphilus', ' g__Aeromonas', ' g__Fusobacterium', ' g__Fenollaria', ' g__Dolosigranulum', ' g__Atopobium', ' g__Finegoldia', ' g__Chryseobacterium', ' g__Alloiococcus', ' g__Leptotrichia', ' g__Jeotgalicoccus', ' g__Gardnerella', ' g__Brachybacterium', ' g__Turicella', ' g__Frederiksenia', ' g__Lactococcus', ' g__Leuconostoc', ' g__Marinomonas', ' g__Pantoea', ' g__Moraxella', ' g__Vibrio', ' g__Blautia'}

Filtered Shared Taxa (>=10% prevalence):
{' g__Pseudomonas', ' g__Rothia', ' g__Gemella', ' g__Kocuria', ' g__Porphyromonas', ' g__Lysobacter',

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_48337/4290181383.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
